# Territory Optimizer — Full Demo

This notebook covers **everything** the project does:
1. **Generate synthetic data** or **upload your own Excel**
2. **Run optimization** via the pipeline
3. **Visualise** results with matplotlib maps & charts
4. **Start the web server** for the interactive Leaflet map UI
5. **Call the REST API** with Python `requests`

---


## 0. Setup

In [ ]:
import logging
logging.basicConfig(level=logging.WARNING, format='%(levelname)s %(message)s')
for lg in ['pipeline', 'optimization', 'features', 'data']:
    logging.getLogger(lg).setLevel(logging.INFO)

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
%matplotlib inline

import os, sys, time, json, threading
from pathlib import Path
from urllib.parse import urljoin

print('Setup OK')

---
## 1A. Generate Synthetic Data

Equivalent to clicking **Generate Data** in the web UI.

In [ ]:
NUM_DEALERS = 300
NUM_FTCS = 15

data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
for f in data_dir.glob('*.parquet'):
    f.unlink()

from data.generate_synthetic_data import SyntheticDataGenerator
gen = SyntheticDataGenerator(num_dealers=NUM_DEALERS, num_ftcs=NUM_FTCS, seed=42)
raw = gen.generate_all(output_dir='data')

print(f'Dealers:       {len(raw["dealers"])}')
print(f'  Static:      {(raw["dealers"]["dealer_type"]=="static").sum()}')
print(f'  Mobile:      {(raw["dealers"]["dealer_type"]=="mobile").sum()}')
print(f'FTCs:          {len(raw["ftcs"])}')
print(f'Relationships: {len(raw["relationships"])}')

### Map: Dealers & FTCs before optimisation

In [ ]:
dealers, ftcs = raw['dealers'], raw['ftcs']

fig, ax = plt.subplots(figsize=(12, 8))

static = dealers[dealers['dealer_type'] == 'static']
mobile = dealers[dealers['dealer_type'] == 'mobile']

ax.scatter(static['dealer_longitude'], static['dealer_latitude'],
           c='blue', s=20, alpha=0.6, label=f'Static ({len(static)})', edgecolors='black', linewidths=0.3)
ax.scatter(mobile['dealer_longitude'], mobile['dealer_latitude'],
           c='cyan', s=15, alpha=0.5, label=f'Mobile ({len(mobile)})', edgecolors='black', linewidths=0.3)
ax.scatter(ftcs['ftc_longitude'], ftcs['ftc_latitude'],
           c='green', s=120, marker='^', label=f'FTCs ({len(ftcs)})', edgecolors='black', linewidths=0.5, zorder=5)

for _, ftc in ftcs.iterrows():
    ax.annotate(ftc['ftc_id'], (ftc['ftc_longitude'], ftc['ftc_latitude']),
                textcoords='offset points', xytext=(0, 10), ha='center', fontsize=7, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='gray', alpha=0.7))

ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('Dealers and FTCs — Before Optimization')
ax.legend(fontsize=9, loc='lower right')
plt.tight_layout(); plt.show()

---
## 1B. Upload Your Own Excel File

Equivalent to the **Upload Excel** toggle in the web UI. Place your `.xlsx` (3 sheets) or `.zip` (3 CSVs) in this directory and run:

In [ ]:
from data.uploader import ExcelUploader

upload_path = 'sample_dataset.xlsx'  # <-- change to your file
if Path(upload_path).exists():
    uploader = ExcelUploader()
    stats = uploader.upload(upload_path)
    print('Upload stats:', stats)
else:
    print(f'No file at {upload_path} — using synthetic data from step 1A')

---
## 2. Run Optimisation

Equivalent to clicking **Run Optimisation** in the web UI.

In [ ]:
from config.settings import ConfigManager
from pipeline import OptimizationPipeline

config = ConfigManager()
pipeline = OptimizationPipeline(config)

result = pipeline.run(parameters={'solver.time_limit_seconds': 10})

print(f'Status:       {result["status"]}')
print(f'Pipeline:     {result["total_pipeline_time"]:.2f}s')
print(f'Solve time:   {result["solve_time"]:.2f}s')
print(f'Active FTCs:  {result["ftcs_used"]}/{result["num_ftcs"]}')
print(f'Reassigned:   {result["changed_dealers"]}/{result["num_dealers"]} ({result["change_rate"]*100:.0f}%)')
print(f'Clusters:     {result["num_clusters"]}')

### Extract assignments for plotting

In [ ]:
from data.loader import DataLoader
from data.processor import DataProcessor
from optimization.clustering import SpatialClusterer
from optimization.model import TerritoryModel

loader = DataLoader()
data_raw = {
    'dealers': loader.load_dealers(),
    'ftcs': loader.load_ftcs(),
    'relationships': loader.load_relationships(),
    'proximity': loader.load_proximity(),
}
proc = DataProcessor()
processed = proc.process_all_data(data_raw)

# Cluster (same as pipeline step 4)
clusterer = SpatialClusterer(config.get_section('clustering'))
processed['cluster_labels'] = clusterer.fit_predict(processed['dealers'], len(processed['ftc_ids']))

# Build features and solve (same as pipeline steps 5-6)
model = TerritoryModel(pipeline._build_run_config({}))
features = pipeline._engineer_features(processed)
solution = model.build_and_solve(features)

new_assign = solution['assignments']
old_assign = solution['current_assignment']
dealer_ids = processed['dealer_ids']
ftc_ids = processed['ftc_ids']

ftc_ids_arr = np.array(ftc_ids)
new_ftc_of_dealer = ftc_ids_arr[new_assign.argmax(axis=1)]
old_ftc_of_dealer = ftc_ids_arr[old_assign.argmax(axis=1)]
changed = new_ftc_of_dealer != old_ftc_of_dealer

print(f'Reassigned: {changed.sum()} / {len(changed)} dealers')
print()
for i, did in enumerate(dealer_ids[:30]):
    if changed[i]:
        print(f'  {did}: {old_ftc_of_dealer[i]} -> {new_ftc_of_dealer[i]}')

### Map: Territories Before vs After

Dealers coloured by their assigned FTC. Green triangles are FTC locations.

In [ ]:
palette = plt.cm.Set1(np.linspace(0, 1, len(ftc_ids)))
ftc_color = dict(zip(ftc_ids, palette))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

for title, ax, assign in [('Before', ax1, old_ftc_of_dealer), ('After', ax2, new_ftc_of_dealer)]:
    for i, row in dealers.iterrows():
        c = ftc_color[assign[i]]
        ax.scatter(row['dealer_longitude'], row['dealer_latitude'],
                   c=[c], s=15, alpha=0.7, edgecolors='black', linewidths=0.2)
    for _, ftc in ftcs.iterrows():
        ax.scatter(ftc['ftc_longitude'], ftc['ftc_latitude'],
                   c='green', s=100, marker='^', edgecolors='black', linewidths=0.5, zorder=5)
        ax.annotate(ftc['ftc_id'], (ftc['ftc_longitude'], ftc['ftc_latitude']),
                    textcoords='offset points', xytext=(0, 10), ha='center', fontsize=7, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='gray', alpha=0.7))
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')

handles = [mpatches.Patch(color=ftc_color[f], label=f, linewidth=0.5, edgecolor='black') for f in ftc_ids]
handles.append(plt.Line2D([0], [0], marker='^', color='w', markerfacecolor='green', markersize=10, label='FTC'))
fig.legend(handles=handles, loc='lower center', ncol=len(ftc_ids) + 1, fontsize=8, framealpha=0.9)
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

### FTC Load Distribution (dealers per FTC)

In [ ]:
old_counts = pd.Series(old_ftc_of_dealer).value_counts().reindex(ftc_ids, fill_value=0)
new_counts = pd.Series(new_ftc_of_dealer).value_counts().reindex(ftc_ids, fill_value=0)

x = np.arange(len(ftc_ids))
w = 0.35

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - w/2, old_counts.values, w, label='Before', color='lightcoral', edgecolor='black', linewidth=0.5)
ax.bar(x + w/2, new_counts.values, w, label='After', color='steelblue', edgecolor='black', linewidth=0.5)
ax.set_xticks(x); ax.set_xticklabels(ftc_ids, fontsize=8)
ax.set_ylabel('Dealers'); ax.set_title('Dealers per FTC')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

### Business Impact

In [ ]:
from analysis.reporting import ReportGenerator
report = ReportGenerator()
print(report.generate_summary_report(result))

---
## 3. Start the Web Server

Runs the Flask server in the background. Access the interactive map UI at **http://localhost:5000** — same as `python3 main.py serve`.

In [ ]:
from api.server import create_app

app = create_app()

server_thread = threading.Thread(target=lambda: app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False), daemon=True)
server_thread.start()
time.sleep(1)

print('Web server running at http://localhost:5000')
print()
print('What you can do from the browser:')
print('  1. Click "Generate Data" to create fresh synthetic data')
print('  2. Toggle "Upload Excel" and upload your .xlsx or .zip')
print('  3. Click "Run Optimization" to solve territories')
print('  4. See red (old) and purple (new) relationship lines on the map')
print('  5. Toggle layers, measure distances, view stats')

### Verify the server is responding

In [ ]:
import requests
BASE = 'http://localhost:5000/api/v1'

r = requests.get(f'{BASE}/health')
print('Health:', r.json())

r = requests.get(f'{BASE}/data/dealers')
dealers_api = r.json()
print(f'Dealers via API: {len(dealers_api)}')

### Run optimisation via REST API

Equivalent to clicking "Run Optimisation" in the web UI.

In [ ]:
r = requests.post(f'{BASE}/optimize', json={'parameters': {'solver.time_limit_seconds': 10}})
api_result = r.json()
print(json.dumps(api_result, indent=2)[:1000])

---
## 4. Generate Sample Excel (for upload testing)

Creates the same `sample_dataset.xlsx` used in the browser UI upload.

In [ ]:
gen = SyntheticDataGenerator(num_dealers=60, num_ftcs=5, seed=42)
raw = gen.generate_all()

dealer_out = pd.DataFrame({
    'DealerID': raw['dealers']['dealer_id'],
    'SM_id': raw['dealers']['sm_id'],
    'dealer_type_static_mobile': raw['dealers']['dealer_type'],
    'AvgCasesPerDay': raw['dealers']['average_cases_per_day'],
    'Count_BFL_Disbursement': raw['dealers']['count_bfl_disbursement'],
    'product_group': raw['dealers']['product_group'],
    'Lat': raw['dealers']['dealer_latitude'],
    'Lng': raw['dealers']['dealer_longitude'],
    'city': raw['dealers']['city'],
})

ftc_out = pd.DataFrame({
    'FTC_ID': raw['ftcs']['ftc_id'],
    'FTE_ID': raw['ftcs']['sm_id'],
    'product_group': raw['ftcs']['product_group'],
    'FTC_Vintage': raw['ftcs']['ftc_vintage'],
    'Count_BFL_Disbursement': raw['ftcs']['count_bfl_disbursement'],
    'AvgCasesPerDay': raw['ftcs']['average_cases_per_day'],
    'PER_SUM_MOB03': raw['ftcs']['per_sum_mob'],
    'NTB_share_per': raw['ftcs']['ntb_share'] * 100,
    'Cross_Sell_share_per': raw['ftcs']['cross_sell'] * 100,
    'Lat': raw['ftcs']['ftc_latitude'],
    'Lng': raw['ftcs']['ftc_longitude'],
    'city': raw['ftcs']['city'],
})

rel_out = pd.DataFrame({
    'FTC_ID': raw['relationships']['ftc_id'],
    'dealer_id': raw['relationships']['dealer_id'],
    'product_category': raw['relationships']['product_category'],
    'AvgCasesPerDay': raw['relationships']['avg_cases_per_day'],
})

output_path = 'sample_dataset.xlsx'
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    dealer_out.to_excel(writer, sheet_name='Dealer Dataset', index=False)
    ftc_out.to_excel(writer, sheet_name='FTC Dataset', index=False)
    rel_out.to_excel(writer, sheet_name='F2D Dataset', index=False)

print(f'Created {output_path}')
print(f'  Dealer Dataset: {len(dealer_out)} rows')
print(f'  FTC Dataset: {len(ftc_out)} rows')
print(f'  F2D Dataset: {len(rel_out)} rows')